# Blend v5 runtime profile (not a submission)

Runs the two blend-v5 children exactly as the submission does (same scored code, same subprocess runner), each on
its own DISJOINT set of 120 training studies posing as the test set (cold page cache for each), and projects the
full ~1,300-study runtime. Model loading is included in each child's time.


In [ ]:
import os, signal, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

T0 = time.time()
WORK = Path("/kaggle/working")
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion",
           "Synovitis", "Baker's", "Contusion", "Fracture"]
# Label priors written by BOTH scored drivers on a failed study (infer_wmil.LABEL_PRIORS == infer_core.LABEL_PRIORS)
PRIOR = np.array([0.1881, 0.0915, 0.3872, 0.2336, 0.2430, 0.1457, 0.3909, 0.3826, 0.3894, 0.1912, 0.2086, 0.1321])
ALPHA = 0.2
PRIOR_TOL_FRAC = 0.005
GRACE_S = 180            # a child still alive this long after its CSV is complete and unchanged is hung: kill it
POLL_S = 15


def _find_dir(root, name, max_depth=3):
    r = Path(root)
    for c in (r / name, r / "competitions" / name, r / "datasets" / name, r / "datasets" / "hrdevelops" / name):
        if c.is_dir():
            return c
    queue = [(r, 0)]
    while queue:
        cur, dep = queue.pop(0)
        if dep < max_depth:
            try:
                for e in cur.iterdir():
                    if e.is_dir():
                        if name in e.name:
                            return e
                        if e.name not in ("test_series", "train_series", "series", "images"):
                            queue.append((e, dep + 1))
            except (PermissionError, OSError):
                pass
    return None


def validate(path: Path, uids: list) -> dict:
    """basic = complete and well-formed; ok = basic and prior-fallback rows within tolerance."""
    r = {"basic": False, "ok": False, "n_prior": None, "why": ""}
    if not path.exists():
        r["why"] = "no CSV"
        return r
    try:
        df = pd.read_csv(path)
    except Exception as e:
        r["why"] = f"unreadable: {type(e).__name__}"
        return r
    if list(df.columns) != ["StudyInstanceUID"] + TARGETS:
        r["why"] = "header mismatch"
        return r
    if len(df) != len(uids) or df["StudyInstanceUID"].duplicated().any() or set(df["StudyInstanceUID"]) != set(uids):
        r["why"] = f"rows/UIDs mismatch ({len(df)} rows vs {len(uids)})"
        return r
    v = df[TARGETS].to_numpy(dtype=float)
    if not np.isfinite(v).all() or (v < 0).any() or (v > 1).any():
        r["why"] = "NaN or out-of-range values"
        return r
    r["basic"] = True
    r["n_prior"] = int(np.all(np.abs(v - PRIOR) < 1e-6, axis=1).sum())
    tol = int(PRIOR_TOL_FRAC * len(df))
    r["ok"] = r["n_prior"] <= tol
    r["why"] = f"{len(df)} rows, {r['n_prior']} on prior (tolerance {tol})"
    return r


def _kill(p):
    for sig, wait in ((signal.SIGTERM, 30), (signal.SIGKILL, 30)):
        try:
            os.killpg(p.pid, sig)
        except ProcessLookupError:
            return
        try:
            p.wait(timeout=wait)
            return
        except subprocess.TimeoutExpired:
            pass


def run_child(name: str, src: str, out_csv: Path, timeout_s: float, uids: list, extra_env=None) -> dict:
    script, log = WORK / f"child_{name}.py", WORK / f"child_{name}.log"
    script.write_text(src)
    if out_csv.exists():
        out_csv.unlink()
    env = dict(os.environ, BLEND_CHILD_OUT=str(out_csv), PYTHONUNBUFFERED="1", **(extra_env or {}))
    t0 = time.time()
    with open(log, "w") as lf:
        p = subprocess.Popen([sys.executable, "-u", str(script)], stdout=lf, stderr=subprocess.STDOUT, env=env,
                             cwd=str(WORK), start_new_session=True)
    pos, seen_mtime, stable_since, end = 0, None, None, None
    print(f"[{name}] started pid {p.pid}, timeout {timeout_s / 3600:.2f} h", flush=True)
    while True:
        rc = p.poll()
        with open(log, errors="replace") as lf:              # stream the child's log into this notebook's log
            lf.seek(pos)
            chunk = lf.read()
            pos = lf.tell()
        for line in chunk.splitlines():
            print(f"  [{name}] {line}", flush=True)
        if rc is not None:
            end = f"exit code {rc}"
            break
        if time.time() - t0 > timeout_s:
            _kill(p)
            end = "TIMEOUT, killed"
            break
        if out_csv.exists() and validate(out_csv, uids)["basic"]:
            m = out_csv.stat().st_mtime
            if m != seen_mtime:
                seen_mtime, stable_since = m, time.time()
            elif time.time() - stable_since > GRACE_S:
                _kill(p)
                end = f"alive {GRACE_S}s after writing a complete CSV, killed"
                break
        time.sleep(POLL_S)
    v = validate(out_csv, uids)
    v.update(name=name, end=end, minutes=(time.time() - t0) / 60)
    print(f"[{name}] {end} after {v['minutes']:.1f} min | CSV: {v['why']} | basic={v['basic']} ok={v['ok']}", flush=True)
    return v

import json
COAT_SRC = 'import importlib.util\nimport os, sys, time\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\n\ndef _find_dir(root: str, name_contains: str, max_depth: int = 3) -> Path | None:\n    root_path = Path(root)\n    if not root_path.exists():\n        return None\n    candidates = [\n        root_path / name_contains,\n        root_path / "competitions" / name_contains,\n        root_path / "datasets" / name_contains,\n        root_path / "datasets" / "hrdevelops" / name_contains,\n    ]\n    for c in candidates:\n        if c.exists() and c.is_dir():\n            return c\n    queue = [(root_path, 0)]\n    while queue:\n        curr, depth = queue.pop(0)\n        if depth > 0 and name_contains in curr.name:\n            return curr\n        if depth < max_depth:\n            try:\n                for entry in curr.iterdir():\n                    if entry.is_dir():\n                        if name_contains in entry.name:\n                            return entry\n                        if entry.name not in ("test_series", "train_series", "series", "images"):\n                            queue.append((entry, depth + 1))\n            except (PermissionError, OSError):\n                pass\n    return None\n\nCODE_DIR = _find_dir("/kaggle/input", "rsna-knee-exp018-code")\nTEST_DIR = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")\nTEST_DIR = Path(os.environ["BLEND_PROFILE_TEST_DIR"])\nWEIGHTS_S42 = _find_dir("/kaggle/input", "rsna-knee-exp018-weights")\nWEIGHTS_S43 = _find_dir("/kaggle/input", "rsna-knee-exp021-weights")\nOUT_PATH = Path(os.environ["BLEND_CHILD_OUT"])\nfor label, v in (("CODE_DIR", CODE_DIR), ("TEST_DIR", TEST_DIR), ("WEIGHTS_S42", WEIGHTS_S42), ("WEIGHTS_S43", WEIGHTS_S43)):\n    print(f"{label}: {v}")\nassert CODE_DIR is not None, "Attach rsna-knee-exp018-code"\nassert TEST_DIR is not None, "Attach the competition data"\nassert WEIGHTS_S42 is not None, "Attach rsna-knee-exp018-weights"\nassert WEIGHTS_S43 is not None, "Attach rsna-knee-exp021-weights"\n\ninit_files = list(CODE_DIR.rglob("__init__.py"))\nassert init_files, f"No __init__.py found under {CODE_DIR}"\npkg_dir = init_files[0].parent\nspec = importlib.util.spec_from_file_location("rsna_knee", pkg_dir / "__init__.py", submodule_search_locations=[str(pkg_dir)])\nrsna_knee = importlib.util.module_from_spec(spec)\nsys.modules["rsna_knee"] = rsna_knee\nspec.loader.exec_module(rsna_knee)\n\nimport timm\nprint("torch", torch.__version__, "timm", timm.__version__, "cuda:", torch.cuda.is_available(),\n      "devices:", torch.cuda.device_count() if torch.cuda.is_available() else 0)\nprint("infer_wmil.py size:", (pkg_dir / "infer_wmil.py").stat().st_size, "B (scored version: 8245 B)")\n\n\nfrom rsna_knee.infer_wmil import run_exp018\nfrom rsna_knee.wmil import TARGET_COLUMNS, WindowMIL\n\n_probe = WindowMIL("coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", pretrained=False)\nprint("arch ok, params %.1fM" % (sum(p.numel() for p in _probe.parameters()) / 1e6))\ndel _probe\n\n# Exactly 5 seed-42 + 5 seed-43 checkpoints, or stop: a missing attachment must never silently turn this\n# into a 5-model run that looks like the ensemble.\nw42 = sorted(WEIGHTS_S42.glob("fold[0-4]_best.pt"))\nw43 = sorted(WEIGHTS_S43.glob("fold[0-4]_seed43_best.pt"))\nprint("seed 42:", [w.name for w in w42])\nprint("seed 43:", [w.name for w in w43])\nassert len(w42) == 5 and len(w43) == 5, f"expected 5 + 5 checkpoints, got {len(w42)} + {len(w43)}"\nweights = w42 + w43\n\nDEVICES = ["cuda:%d" % i for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else ["cpu"]\nDECODE_WORKERS = max(1, min((os.cpu_count() or 4) - 1, 8))\nBUDGET_SECONDS = 8.0 * 3600 * 0.85     # leave margin under the 9h Kaggle cap; the guard drops models, never studies\nprint("devices:", DEVICES, "decode workers:", DECODE_WORKERS, "checkpoints:", len(weights))\n\n\nt0 = time.time()\nsub_df = run_exp018(\n    TEST_DIR / "test.csv", TEST_DIR / "test_series.csv", TEST_DIR / "test_series", weights, OUT_PATH,\n    devices=DEVICES, num_decode_workers=DECODE_WORKERS, budget_seconds=BUDGET_SECONDS,\n)\nprint("total runtime %.1f min" % ((time.time() - t0) / 60))\n\n\ntest_df = pd.read_csv(TEST_DIR / "test.csv")\nsub_df = pd.read_csv(OUT_PATH)[["StudyInstanceUID"] + TARGET_COLUMNS]\nassert len(sub_df) == len(test_df), f"row count mismatch: {len(sub_df)} vs {len(test_df)}"\nassert set(sub_df["StudyInstanceUID"]) == set(test_df["StudyInstanceUID"]), "study UID mismatch"\nassert not sub_df[TARGET_COLUMNS].isna().any().any(), "NaNs in submission"\nassert (sub_df[TARGET_COLUMNS] >= 0).all().all() and (sub_df[TARGET_COLUMNS] <= 1).all().all(), "probabilities out of [0,1]"\nassert list(sub_df.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS, f"header mismatch: {list(sub_df.columns)}"\nsub_df.to_csv(OUT_PATH, index=False)\nprint(f"Wrote {OUT_PATH}: {len(sub_df)} rows, all checks passed.")\nsub_df.head()\n\n'
SLOT_SRC = 'import importlib.util\nimport os, sys, time, traceback\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\ndef _find_dir(root: str, name_contains: str, max_depth: int = 3) -> Path | None:\n    root_path = Path(root)\n    if not root_path.exists():\n        return None\n    \n    # 1. Fast direct path checks for standard Kaggle layouts\n    candidates = [\n        root_path / name_contains,\n        root_path / "competitions" / name_contains,\n        root_path / "datasets" / name_contains,\n        root_path / "datasets" / "hrdevelops" / name_contains,\n    ]\n    for c in candidates:\n        if c.exists() and c.is_dir():\n            return c\n\n    # 2. Fast shallow BFS (up to 3 levels, skipping image series trees)\n    queue = [(root_path, 0)]\n    while queue:\n        curr, depth = queue.pop(0)\n        if depth > 0 and name_contains in curr.name:\n            return curr\n        if depth < max_depth:\n            try:\n                for entry in curr.iterdir():\n                    if entry.is_dir():\n                        if name_contains in entry.name:\n                            return entry\n                        if entry.name not in ("test_series", "train_series", "series", "images"):\n                            queue.append((entry, depth + 1))\n            except (PermissionError, OSError):\n                pass\n    return None\n\nCODE_DIR = _find_dir("/kaggle/input", "rsna-knee-code-0916")\nTEST_DIR = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")\nTEST_DIR = Path(os.environ["BLEND_PROFILE_TEST_DIR"])\nOUT_PATH = Path(os.environ["BLEND_CHILD_OUT"])\n\nprint("CODE_DIR:", CODE_DIR)\nprint("TEST_DIR:", TEST_DIR)\nassert CODE_DIR is not None, "Attach rsna-knee-code"\nassert TEST_DIR is not None, "Attach the competition data"\n\ninit_files = list(CODE_DIR.rglob("__init__.py"))\nassert init_files, f"No __init__.py found under {CODE_DIR}"\npkg_dir = init_files[0].parent\nspec = importlib.util.spec_from_file_location("rsna_knee", pkg_dir / "__init__.py", submodule_search_locations=[str(pkg_dir)])\nrsna_knee = importlib.util.module_from_spec(spec)\nsys.modules["rsna_knee"] = rsna_knee\nspec.loader.exec_module(rsna_knee)\n\nfrom rsna_knee.dataset import TARGET_COLUMNS, _patch_size_for_backbone\nfrom rsna_knee.infer_core import (\n    WallClockGuard, decode_one_for_pool, ensemble_blend, load_ensemble_recipe, nearest_patch_multiple,\n    prefetch_decode, prior_row, run_study_inference, run_study_inference_tta,\n)\nfrom rsna_knee.model import SlotModel\n\n# Frozen recipe (src/rsna_knee/ensemble_recipe_v1.yaml, bundled in the rsna-knee-code dataset) —\n# members, per-label blend method, and the TTA decision all come from THIS file, not a hardcoded\n# list here, so a future recipe version is a new file + one path change, not a notebook edit.\nRECIPE_PATH = pkg_dir / "ensemble_recipe_v1.yaml"\nassert RECIPE_PATH.exists(), f"Recipe file not found at {RECIPE_PATH} — check the rsna-knee-code dataset version"\nloaded = load_ensemble_recipe(RECIPE_PATH)\nMEMBERS = {name: {**spec} for name, spec in loaded["members"].items()}\nRECIPE = loaded["recipe"]  # {label: (members_list, method)}\nTTA_ACCEPTED = loaded["tta_enabled"]\nfor name, spec in MEMBERS.items():\n    spec["dir"] = _find_dir("/kaggle/input", spec["kaggle_dataset"])\n    print(name, "->", spec["dir"])\n    assert spec["dir"] is not None and spec["dir"].exists(), f"Attach {spec[\'kaggle_dataset\']}"\n\nprint(f"Recipe version {loaded[\'raw\'][\'version\']} ({loaded[\'raw\'][\'created\']}), "\n      f"TTA_ACCEPTED={TTA_ACCEPTED}")\nprint("torch", torch.__version__, "cuda available:", torch.cuda.is_available(),\n      "device count:", torch.cuda.device_count() if torch.cuda.is_available() else 0)\n\nSLICES_PER_SERIES = 24\nIMAGE_SIZE = 384\nOUT_SIZE = nearest_patch_multiple(IMAGE_SIZE, _patch_size_for_backbone(next(iter(MEMBERS.values()))["backbone"]))  # 378 — same for all 3 members (all patch14)\n\nGLOBAL_MEMBERS = list(MEMBERS.keys())\n\n# fp16, not bf16 (2026-09-11 T4 adaptation): the pinned Kaggle accelerator is T4 x2 — T4 is sm_75\n# (Turing), with no native bf16 tensor-core support; bf16 autocast on it either falls back to a\n# slow/emulated path or errors depending on the torch build. The efficiency notebook already used\n# fp16 for exactly this "smaller/older GPU" reason (cell-0 there); infer_core.run_study_inference[_tta]\n# still DEFAULT to bf16 (correct for H100-based HPC rehearsal/dev, left untouched) — this notebook\n# now passes fp16 explicitly at every call site instead of relying on that default.\nAUTOCAST_DTYPE = torch.float16\n\n# Decode overlap (2026-09-11): studies are decoded in a background thread pool (infer_core.\n# prefetch_decode) while the GPU works the previous study, instead of the old decode-study-N-then-\n# infer-study-N lockstep that left the CPU idle during every inference and the GPU idle during\n# every decode. Worker count follows the same "trust os.cpu_count() on Kaggle" reasoning as the\n# efficiency notebook (that cell\'s comment has the full 192-vs-16 HPC-SLURM backstory).\nDECODE_WORKERS = max(1, min((os.cpu_count() or 4) - 1, 32))\n\nBUDGET_SECONDS = 8 * 3600  # 9h hard cap (CLAUDE.md §1) minus margin for scheduling/write\nCALIBRATION_STUDIES = 10   # measure actual per-study cost on this many before picking a ladder rung\n\nDEVICE = "cpu"\nif torch.cuda.is_available():\n    major, minor = torch.cuda.get_device_capability(0)\n    cap = major * 10 + minor\n    supported_caps = [int(a[3:]) for a in torch.cuda.get_arch_list() if a.startswith("sm_")]\n    if supported_caps and cap >= min(supported_caps):\n        DEVICE = "cuda"\n    else:\n        print(f"  [WARN] GPU {torch.cuda.get_device_name(0)} (sm_{cap}) unsupported by this torch "\n              f"build (min sm_{min(supported_caps) if supported_caps else \'?\'}) — CPU fallback.")\n\n# Multi-GPU member sharding (2026-09-11, T4 x2): each member\'s 5 folds run on ONE assigned device,\n# and different members\' devices are driven from separate Python threads (cell-5) so their CUDA\n# kernels actually overlap instead of queuing behind each other on a single device — PyTorch\'s C++\n# backend releases the GIL during a CUDA call, so two threads targeting cuda:0/cuda:1 get genuine\n# concurrent GPU execution, not just concurrent Python bookkeeping. Falls back to everything on one\n# device (today\'s behavior) when only one GPU is visible (e.g. a normal single-GPU HPC job).\nN_GPUS = torch.cuda.device_count() if DEVICE == "cuda" else 0\n\n\ndef _assign_member_devices(members: dict, n_gpus: int) -> dict:\n    if n_gpus <= 1:\n        return {name: DEVICE for name in members}\n    loads = [0.0] * n_gpus\n    assignment = {}\n    # Heaviest (ViT-B, cost 3) first so the greedy balance doesn\'t strand a big model behind an\n    # already-loaded GPU — same relative-cost weighting as relative_cost() below.\n    for name, spec in sorted(members.items(),\n                              key=lambda kv: -(3.0 if kv[1]["backbone"].startswith("vit_base") else 1.0)):\n        cost = 3.0 if spec["backbone"].startswith("vit_base") else 1.0\n        gpu = min(range(n_gpus), key=lambda i: loads[i])\n        assignment[name] = f"cuda:{gpu}"\n        loads[gpu] += cost\n    return assignment\n\n\nMEMBER_DEVICE = _assign_member_devices(MEMBERS, N_GPUS)\nprint("device:", DEVICE, "n_gpus:", N_GPUS, "out_size:", OUT_SIZE, "decode_workers:", DECODE_WORKERS,\n      "TTA_ACCEPTED:", TTA_ACCEPTED)\nprint("member device assignment:", MEMBER_DEVICE)\n\ndef load_member_models(spec: dict, device: str) -> list:\n    # Weight datasets are staged FLAT (foldN_best.pt at the dataset root, no subdirectories,\n    # 2026-09-11) specifically to avoid a documented Kaggle risk: uploading per-fold\n    # SUBDIRECTORIES requires --dir-mode zip (plain upload silently DROPS subdirectories\n    # entirely), and the 2026-09-08 Kaggle debugging arc (EXPERIMENTS.md) found that zip-based\n    # subdirectory upload does not reliably preserve folder nesting on extraction — fine for the\n    # code package (unique .py filenames, just find __init__.py wherever it lands) but every\n    # fold\'s checkpoint is identically named "best.pt", so a flattening extraction could silently\n    # collide 5 files into 1. Falls back to the foldN/best.pt subdirectory layout for robustness\n    # in case a dataset is ever staged that way instead.\n    ckpt_paths = sorted(spec["dir"].glob("fold*_best.pt"))\n    if not ckpt_paths:\n        ckpt_paths = [d / "best.pt" for d in sorted(spec["dir"].glob("fold*"))\n                      if d.is_dir() and (d / "best.pt").exists()]\n    models = []\n    for ckpt_path in ckpt_paths:\n        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)\n        model = SlotModel(spec["backbone"], num_targets=len(TARGET_COLUMNS), pretrained=False).to(device)\n        model.load_state_dict(ckpt["ema"])\n        model.eval()\n        models.append(model)\n    return models\n\nmember_models, member_rank_refs = {}, {}\nfor name, spec in MEMBERS.items():\n    member_device = MEMBER_DEVICE[name]\n    member_models[name] = load_member_models(spec, member_device)\n    npz_path = spec["dir"] / "rank_reference.npz"\n    member_rank_refs[name] = dict(np.load(npz_path)) if npz_path.exists() else {}\n    print(f"{name} -> {member_device}: {len(member_models[name])}/5 folds loaded, rank_reference "\n          f"{\'found\' if npz_path.exists() else \'MISSING (rank-mean labels fall back to prob-mean)\'}")\n\nif all(len(m) == 0 for m in member_models.values()):\n    raise RuntimeError("No member models loaded at all — cannot run inference.")\n\ntest_df = pd.read_csv(TEST_DIR / "test.csv")\nseries_df = pd.read_csv(TEST_DIR / "test_series.csv").rename(\n    columns={"StudyInstanceUID": "study_uid", "SeriesInstanceUID": "series_uid"}\n)\nseries_root = TEST_DIR / "test_series"\nprint(f"{len(test_df)} test studies, {len(series_df)} series.")\n\n# Ladder rungs, cheapest-fallback last. Each rung = (use_tta, members_to_use). TTA is measured\n# OOF-negative (frozen recipe file) so it never appears as an active rung, only smaller member sets.\nLADDER = [\n    ("no-TTA + 3 members", False, GLOBAL_MEMBERS),\n    ("no-TTA + 2 members", False, ["exp006_dinov2b_slot_v3labels", "exp005_dinov2s_slot_v3labels"]),\n    ("smallest model only", False, ["exp005_dinov2s_slot_v3labels"]),\n]\n\n# Members run in separate threads (one per member, cell-2\'s MEMBER_DEVICE says which GPU each\n# lands on) so members on DIFFERENT devices actually execute concurrently — members sharing a\n# device just serialize on that device\'s stream as before, no loss either way. max_workers=3 (not\n# N_GPUS) since two members can share a device without harm; the thread pool itself is cheap.\n_member_pool = ThreadPoolExecutor(max_workers=max(1, len(GLOBAL_MEMBERS)))\n\n\ndef relative_cost(use_tta: bool, members: list[str]) -> float:\n    """Rough relative inference cost vs. one ViT-S forward pass — ViT-B counted as 3x ViT-S\n    (roughly matches parameter-count scaling), TTA doubles whichever members run."""\n    unit = sum(3.0 if MEMBERS[m]["backbone"].startswith("vit_base") else 1.0 for m in members)\n    return unit * (2.0 if use_tta else 1.0)\n\n\ndef run_member(name: str, slots, tag_sides, use_tta: bool) -> np.ndarray:\n    models = member_models[name]\n    device = MEMBER_DEVICE[name]\n    if not models:\n        return prior_row()\n    if use_tta:\n        return run_study_inference_tta(slots, tag_sides, models, device, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)\n    return run_study_inference(slots, tag_sides, models, device, SLICES_PER_SERIES, OUT_SIZE, AUTOCAST_DTYPE)\n\n\ndef run_ensemble(slots, tag_sides, use_tta: bool, members: list[str]) -> np.ndarray:\n    # Submit every member\'s forward pass before blocking on any result — with members split across\n    # cuda:0/cuda:1 (T4 x2) this is what actually gets both GPUs busy at once (2026-09-11 T4\n    # adaptation) instead of finishing one member\'s device-to-device-synchronous call before\n    # starting the next.\n    futures = {m: _member_pool.submit(run_member, m, slots, tag_sides, use_tta) for m in members}\n    member_probs = {m: f.result() for m, f in futures.items()}\n    # RECIPE (from the frozen recipe file) already gives {label: (members, method)} for the full\n    # 3-member set; when the ladder has dropped to fewer members, use that label\'s method but\n    # restrict to whichever of those members are actually active at this rung.\n    active_recipe = {label: ([m for m in orig_members if m in members], method)\n                      for label, (orig_members, method) in RECIPE.items()}\n    return ensemble_blend(member_probs, active_recipe, member_rank_refs)\n\nimport gc\n\nguard = WallClockGuard(BUDGET_SECONDS)\n\n\ndef _study_task(study_uid: str) -> tuple:\n    study_series = series_df[series_df["study_uid"] == study_uid].to_dict("records")\n    return (study_uid, study_series, str(series_root), SLICES_PER_SERIES, OUT_SIZE)\n\n\n# --- Calibration: run the full (top) rung on the first few studies through the SAME decode-\n# overlap pipeline as the main loop (prefetch_decode — CPU decode of study N+1 runs in a\n# background thread pool while the GPU is still working study N, 2026-09-11 T4 adaptation), so\n# the measured per-study cost reflects the actual overlapped pipeline, not a serial one. Project\n# total time for every rung, pick the highest one that fits with margin — one rung is used for the\n# WHOLE run (not switched mid-way) so every study is scored consistently.\nn_calib = min(CALIBRATION_STUDIES, len(test_df))\ncalib_uids = list(test_df["StudyInstanceUID"].iloc[:n_calib])\ncalib_tasks = [_study_task(u) for u in calib_uids]\n\ncalib_t0 = time.time()\ncalib_rows = []\nfor study_uid, slots, tag_sides, err in prefetch_decode(calib_tasks, decode_one_for_pool, DECODE_WORKERS):\n    if slots and err is None:\n        # inference_mode also applied via infer_core._run_orientation\'s own decorator\n        # (thread-local, so it must live on the actual forward-pass function to cover\n        # _member_pool\'s worker threads) -- wrapped here too for an explicit, auditable\n        # no-autograd-state guarantee at every call site (2026-09-16 OOM fix).\n        with torch.inference_mode():\n            probs = run_ensemble(slots, tag_sides, *LADDER[0][1:])\n        calib_rows.append({"StudyInstanceUID": study_uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, probs)}})\n        del slots\ncalib_elapsed = time.time() - calib_t0\nper_study_at_rung0 = calib_elapsed / max(n_calib, 1)\nrung0_cost = relative_cost(*LADDER[0][1:])\nprint(f"Calibration: {n_calib} studies, {calib_elapsed:.1f}s ({per_study_at_rung0:.2f}s/study at rung 0, decode-overlapped)")\n\nchosen_rung_idx = 0\nfor idx, (label, use_tta, members) in enumerate(LADDER):\n    projected_per_study = per_study_at_rung0 * (relative_cost(use_tta, members) / rung0_cost)\n    projected_total = projected_per_study * len(test_df)\n    fits = projected_total <= guard.budget * 0.85  # 15% margin under budget\n    print(f"  rung {idx} ({label}): projected {projected_total / 60:.1f} min for {len(test_df)} studies "\n          f"-> {\'FITS\' if fits else \'too slow\'}")\n    if fits:\n        chosen_rung_idx = idx\n        break\nelse:\n    chosen_rung_idx = len(LADDER) - 1  # nothing fit comfortably — use the cheapest rung anyway\n\nrung_label, USE_TTA, ACTIVE_MEMBERS = LADDER[chosen_rung_idx]\nprint(f"Chosen rung: {chosen_rung_idx} ({rung_label})")\n\nrows = list(calib_rows) if chosen_rung_idx == 0 else []  # reuse calibration results only if they match the chosen rung\ndone_uids = {r["StudyInstanceUID"] for r in rows}\nremaining_uids = [u for u in test_df["StudyInstanceUID"] if u not in done_uids]\nremaining_tasks = [_study_task(u) for u in remaining_uids]\n\nn_prior_fallback = n_dual_flip = 0\nprocessed = len(rows)\n# Guard is checked at the TOP of each iteration, before touching the item prefetch_decode just\n# handed back — if it trips, we stop consuming the generator and bulk-fill everything left with\n# priors in one shot (cheaper than looping through each remaining study individually). The\n# already-decoded item that triggered the check is simply discarded. Note: prefetch_decode\'s\n# background feeder thread (daemon=True) may still be mid-decode on a few more studies at the\n# moment we stop consuming — harmless (wasted CPU, no correctness impact), dies with the kernel;\n# not worth generator-cancellation machinery for a rare, graceful-degradation path in a one-shot\n# submission notebook.\nfor study_uid, slots, tag_sides, err in prefetch_decode(remaining_tasks, decode_one_for_pool, DECODE_WORKERS):\n    if guard.over(margin=0.05):\n        break\n    row = {"StudyInstanceUID": study_uid}\n    try:\n        if err is not None or not slots:\n            raise RuntimeError(err or "no series decoded successfully for this study")\n        if all(s not in ("R", "L") for s in tag_sides):\n            n_dual_flip += 1\n        with torch.inference_mode():\n            probs = run_ensemble(slots, tag_sides, USE_TTA, ACTIVE_MEMBERS)\n        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, probs)})\n    except Exception as e:\n        print(f"  [WARN] {study_uid}: {type(e).__name__}: {e} — using label priors")\n        n_prior_fallback += 1\n        row.update({c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())})\n    del slots  # decoded volumes are the biggest per-study allocation (up to 6 slots x\n    # T x out_size x out_size float32 each) -- drop the reference as soon as this\n    # study\'s row is built, don\'t wait for the next loop iteration\'s reassignment\n    rows.append(row)\n    processed += 1\n    if processed % 25 == 0:\n        gc.collect()  # 2026-09-16 Kaggle OOM fix: reclaim decoded-volume memory on a\n        # fixed cadence rather than trusting refcounting + the allocator alone over a\n        # ~1300-study run\n    if processed % 100 == 0 or processed == len(test_df):\n        print(f"  {processed}/{len(test_df)} studies, {guard.elapsed():.0f}s elapsed ({guard.fraction_used():.1%} of budget)")\n\nif processed < len(test_df):\n    done_now = {r["StudyInstanceUID"] for r in rows}\n    skipped = [u for u in remaining_uids if u not in done_now]\n    print(f"  [BUDGET] {guard.elapsed():.0f}s used, {processed}/{len(test_df)} done — priors for the rest.")\n    for uid in skipped:\n        rows.append({"StudyInstanceUID": uid, **{c: float(p) for c, p in zip(TARGET_COLUMNS, prior_row())}})\n        n_prior_fallback += 1\n\nprint(f"Done: {len(rows)} studies, rung={chosen_rung_idx} ({rung_label}), {n_prior_fallback} full-prior fallbacks, "\n      f"{n_dual_flip} dual-flip, {guard.elapsed():.0f}s total.")\n\nsub_df = pd.DataFrame(rows)[["StudyInstanceUID"] + TARGET_COLUMNS]\n\nassert len(sub_df) == len(test_df), f"row count mismatch: {len(sub_df)} vs {len(test_df)}"\nassert set(sub_df["StudyInstanceUID"]) == set(test_df["StudyInstanceUID"]), "study UID mismatch"\nassert not sub_df[TARGET_COLUMNS].isna().any().any(), "NaNs in submission"\nassert (sub_df[TARGET_COLUMNS] >= 0).all().all() and (sub_df[TARGET_COLUMNS] <= 1).all().all(), "probabilities out of [0,1]"\nassert list(sub_df.columns) == ["StudyInstanceUID"] + TARGET_COLUMNS, f"header mismatch: {list(sub_df.columns)}"\n\nsub_df.to_csv(OUT_PATH, index=False)\nprint(f"Wrote {OUT_PATH}: {len(sub_df)} rows, all checks passed. Rung used: {chosen_rung_idx} ({rung_label}).")\nsub_df.head()\n'
COMP = _find_dir("/kaggle/input", "rsna-knee-abnormality-detection")
sdf = pd.read_csv(COMP / "train_series.csv")
uids = sorted(sdf["StudyInstanceUID"].unique())
SETS = {"coat": uids[500:620], "slot": uids[620:740]}   # disjoint from each other and from earlier profiling runs


In [ ]:
R = {}
for name, src in (("coat", COAT_SRC), ("slot", SLOT_SRC)):
    us = SETS[name]
    d = Path("/tmp") / f"_fake_{name}"; d.mkdir(exist_ok=True)   # not under /kaggle/working: outputs must not follow the symlink
    pd.DataFrame({"StudyInstanceUID": us}).to_csv(d / "test.csv", index=False)
    sdf[sdf.StudyInstanceUID.isin(us)].to_csv(d / "test_series.csv", index=False)
    if not (d / "test_series").exists():
        (d / "test_series").symlink_to(COMP / "train_series")
    v = run_child(name, src, WORK / f"preds_{name}.csv", 3 * 3600, us, {"BLEND_PROFILE_TEST_DIR": str(d)})
    R[name] = {"n": len(us), "minutes": v["minutes"], "s_per_study": v["minutes"] * 60 / len(us),
               "min_per_1300": v["minutes"] / len(us) * 1300, "end": v["end"], "csv": v["why"], "ok": v["ok"]}
R["total_min_per_1300"] = R["coat"]["min_per_1300"] + R["slot"]["min_per_1300"]
R["total_h_per_1300"] = R["total_min_per_1300"] / 60
print(json.dumps(R, indent=1))
(WORK / "profile_blend_v5.json").write_text(json.dumps(R, indent=1))
assert "torch" not in sys.modules
